<a href="https://colab.research.google.com/github/Flguima/PROJETOS/blob/main/MOD41_EXERCICIO_DUELOFINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ------------------------------------------------------------
# DUEL0 ENTRE SVM E XGBOOST - COMPETIÇÃO TITANIC (Kaggle)
# ------------------------------------------------------------
# Objetivo: comparar o desempenho dos modelos SVM e XGBoost
# aplicando técnicas de pré-processamento, balanceamento,
# busca de hiperparâmetros e validação cruzada.
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 1. Upload dos arquivos para o Colab
# ------------------------------------------------------------
# No Colab, precisamos enviar os arquivos locais (train.csv, test.csv, gender_submission.csv).
from google.colab import files
uploaded = files.upload()

# 2. Carregamento dos dados
# ------------------------------------------------------------
# Carregamos os datasets Titanic.
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
gender_submission = pd.read_csv("gender_submission.csv")

print("Dados carregados com sucesso!")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

# 3. Pré-processamento
# ------------------------------------------------------------
# Justificativas:
# - Age e Fare: usamos mediana para preencher valores nulos (robusta contra outliers).
# - Embarked: usamos moda (categoria mais frequente).
# - Variáveis categóricas (Sex, Embarked) transformadas em dummies.
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(test['Age'].median())
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

train = pd.get_dummies(train, columns=['Sex','Embarked'], drop_first=True)
test = pd.get_dummies(test, columns=['Sex','Embarked'], drop_first=True)

features = ['Pclass','Age','SibSp','Parch','Fare','Sex_male','Embarked_Q','Embarked_S']
X = train[features]
y = train['Survived']
X_test = test[features]

print("Pré-processamento concluído!")

# 4. Balanceamento de classes
# ------------------------------------------------------------
# Justificativa:
# - O dataset Titanic é desbalanceado (mais mortos que sobreviventes).
# - Usamos SMOTE para gerar exemplos sintéticos da classe minoritária.
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print("Balanceamento concluído! Classes após SMOTE:", np.bincount(y_res))

# 5. Definição dos modelos
# ------------------------------------------------------------
# Justificativa:
# - SVM: bom para datasets pequenos e margens claras.
# - XGBoost: poderoso em dados tabulares, lida bem com não-linearidades.
svm = SVC(probability=True, random_state=42)
xgb = XGBClassifier(eval_metric='logloss', random_state=42)

pipe_svm = Pipeline([('scaler', StandardScaler()), ('svm', svm)])
pipe_xgb = Pipeline([('xgb', xgb)])

# 6. Busca de hiperparâmetros
# ------------------------------------------------------------
# Justificativa:
# - Usamos GridSearchCV para testar combinações de parâmetros.
# - scoring='accuracy' porque Kaggle avalia acurácia.
param_svm = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['linear','rbf'],
    'svm__gamma': ['scale','auto']
}
grid_svm = GridSearchCV(pipe_svm, param_svm, cv=5, scoring='accuracy')
grid_svm.fit(X_res, y_res)

param_xgb = {
    'xgb__n_estimators': [100,200,300],
    'xgb__max_depth': [3,5,7],
    'xgb__learning_rate': [0.01,0.1,0.2],
    'xgb__subsample': [0.8,1]
}
grid_xgb = GridSearchCV(pipe_xgb, param_xgb, cv=5, scoring='accuracy')
grid_xgb.fit(X_res, y_res)

# 7. Avaliação com Cross Validation
# ------------------------------------------------------------
# Justificativa:
# - Usamos cross_val_score para validar robustez do modelo.
scores_svm = cross_val_score(grid_svm.best_estimator_, X_res, y_res, cv=5)
scores_xgb = cross_val_score(grid_xgb.best_estimator_, X_res, y_res, cv=5)

print("SVM - Score médio CV:", scores_svm.mean())
print("XGBoost - Score médio CV:", scores_xgb.mean())

# 8. Escolha do vencedor e submissão
# ------------------------------------------------------------
# Justificativa:
# - Comparamos scores médios.
# - O melhor modelo gera o arquivo submission.csv para Kaggle.
best_model = grid_xgb if scores_xgb.mean() > scores_svm.mean() else grid_svm
winner = "XGBoost" if best_model == grid_xgb else "SVM"
print("Modelo vencedor:", winner)

predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions
})
submission.to_csv("submission.csv", index=False)

print("Arquivo submission.csv gerado com sucesso!")

# 9. Gerar arquivo com resultado do duelo + explicações
# ------------------------------------------------------------
# Justificativa:
# - Criamos um relatório textual com explicações e resultados numéricos.
with open("duelo_resultados.txt", "w") as f:
    f.write("Resultados do Duelo Titanic\n")
    f.write("----------------------------\n\n")
    f.write("Explicações e justificativas:\n")
    f.write("- Valores nulos tratados com mediana (Age/Fare) e moda (Embarked).\n")
    f.write("- Variáveis categóricas transformadas em dummies para uso nos modelos.\n")
    f.write("- SMOTE usado para balancear classes e evitar viés.\n")
    f.write("- SVM avaliado com normalização via StandardScaler.\n")
    f.write("- XGBoost avaliado com parâmetros ajustados via GridSearchCV.\n\n")
    f.write("Resultados numéricos (17 casas decimais):\n")
    f.write(f"SVM - Score médio CV: {scores_svm.mean():.17f}\n")
    f.write(f"XGBoost - Score médio CV: {scores_xgb.mean():.17f}\n")
    f.write(f"Modelo vencedor: {winner}\n")

print("Arquivo duelo_resultados.txt gerado com sucesso!")

# 10. Download automático dos arquivos
# ------------------------------------------------------------
files.download("submission.csv")
files.download("duelo_resultados.txt")

Saving gender_submission.csv to gender_submission (2).csv
Saving test.csv to test (2).csv
Saving train.csv to train (2).csv
Dados carregados com sucesso!
Train shape: (891, 12)
Test shape: (418, 11)
Pré-processamento concluído!
Balanceamento concluído! Classes após SMOTE: [549 549]
SVM - Score médio CV: 0.7923868825238688
XGBoost - Score médio CV: 0.8242507264425072
Modelo vencedor: XGBoost
Arquivo submission.csv gerado com sucesso!
Arquivo duelo_resultados.txt gerado com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>